In [ ]:
# ---------------- Imports ----------------
import os
import json
import random
import yaml
from collections import defaultdict
import csv
import pandas as pd
import re



In [ ]:
# ---------------- Args ----------------
SOURCE_CHOICE = "20260115T095923-combined-claims-full-40-all-batches"
RESULTS_CHOICE = "019c408d-bdfe-7734-bddb-2f2a8d0ab953"

ANNOTATOR_ID_RETURNED = [
    "60e197c98498a8f2baa8bc24",
    "691371c5d552995298408398",
    "65944a7812155ed118b836c4",
]



In [ ]:
# ---------------- Config ----------------
with open("../../config/config.yaml", "r") as f:
    config = yaml.safe_load(f)

PROJ_STORE = config["paths"]["proj-store"]
HUMAN_EVAL_PATH = os.path.join(PROJ_STORE, "evaluation", "human-evaluation")

SOURCE_PATH = os.path.join(HUMAN_EVAL_PATH, "raw-batch", "batches_merged", SOURCE_CHOICE)
RESULTS_PATH = os.path.join(HUMAN_EVAL_PATH, "results", "raw", RESULTS_CHOICE)

# Results
OUTPUT_PATH = os.path.join(HUMAN_EVAL_PATH, "results", "processed")
os.makedirs(OUTPUT_PATH, exist_ok=True)
OUTPUT_FILEPATH = os.path.join(OUTPUT_PATH, f"{RESULTS_CHOICE}-processed.csv")




In [ ]:
# ---------------- Functions ----------------
def find_unique_full_claim_id(text):
    if text not in source_grouped:
        raise ValueError(f"No match found in SOURCE for text:\n{text}")

    matches = source_grouped[text]

    if len(matches) > 1:
        raise ValueError(
            f"Multiple matches found in SOURCE for text:\n{text}\n"
            f"Matched full_claim_id values: {matches}"
        )

    return matches[0]






In [ ]:
# ---------------- Run ----------------

# Load CSVs
source_df = pd.read_csv(f"{SOURCE_PATH}.csv")
results_df = pd.read_csv(f"{RESULTS_PATH}.csv")

# Normalize column name in RESULTS
if "Claim" in results_df.columns:
    results_df = results_df.rename(columns={"Claim": "text"})

# ---------------- Check for exact duplicate rows in RESULTS ----------------

duplicate_rows = results_df[results_df.duplicated(keep=False)]

if not duplicate_rows.empty:
    raise ValueError(
        "Exact duplicate rows found in RESULTS:\n"
        + duplicate_rows.to_string(index=False)
    )

# ---------------- Sanity checks ----------------

required_source_cols = {"text", "full_claim_id", "framing_type", "true_label"}
missing = required_source_cols - set(source_df.columns)
if missing:
    raise ValueError(f"SOURCE file is missing columns: {missing}")

if "text" not in results_df.columns:
    raise ValueError("RESULTS file is missing 'text' column")

# ---------------- Row-by-row exact match lookup ----------------

def find_unique_source_match(text):
    matches = source_df[source_df["text"] == text]

    if len(matches) == 0:
        raise ValueError(
            f"No match found in SOURCE for text:\n{text}"
        )

    if len(matches) > 1:
        raise ValueError(
            f"Multiple matches found in SOURCE for text:\n{text}\n"
            f"Matching full_claim_id values: {matches['full_claim_id'].tolist()}"
        )

    row = matches.iloc[0]
    return pd.Series({
        "full_claim_id": row["full_claim_id"],
        "framing_type": row["framing_type"],
        "true_label": row["true_label"],
    })

# ---------------- Apply lookup ----------------

results_df[["full_claim_id", "framing_type", "true_label"]] = (
    results_df["text"].apply(find_unique_source_match)
)

# ---------------- Final table ----------------

display(results_df.head())

results_df.to_csv(OUTPUT_FILEPATH, index=False)


In [ ]:
# Identify annotator indices dynamically 
annotator_nums = sorted(
    {
        int(re.search(r"Annotator(\d+)_", c).group(1))
        for c in results_df.columns
        if re.search(r"Annotator(\d+)_", c)
    }
)

rows = []

for _, row in results_df.iterrows():
    for i in annotator_nums:
        annotator_id_col = f"Annotator{i}_ID"
        response_col = f"Annotator{i}_Response"
        timestamp_col = f"Annotator{i}_Timestamp"

        if pd.isna(row.get(annotator_id_col)):
            continue  # annotator did not answer

        rows.append({
            "DataPoint_ID": row["DataPoint_ID"],
            "Task_Group_ID": row["Task_Group_ID"],
            "Task_Type": row["Task_Type"],
            "Question": row["Question"],
            "text": row["text"],
            "Annotator_ID": row[annotator_id_col],
            "Response": row[response_col],
            "Timestamp": row[timestamp_col],
            "full_claim_id": row["full_claim_id"],
            "framing_type": row["framing_type"],
            "true_label": row["true_label"],
        })

long_df = pd.DataFrame(rows)

question_map = {
    "Is this accurate?": "accurate_response",
    "How confident are you in your answer?": "confidence_response",
    "How familiar are you with this topic?": "familiarity_response",
}

long_df["question_key"] = long_df["Question"].map(question_map)

if long_df["question_key"].isna().any():
    bad = long_df.loc[long_df["question_key"].isna(), "Question"].unique()
    raise ValueError(f"Unexpected question text found: {bad}")

tidy_df = (
    long_df
    .pivot_table(
        index=[
            "DataPoint_ID",
            "Task_Group_ID",
            "Task_Type",
            "text",
            "Annotator_ID",
            "full_claim_id",
            "framing_type",
            "true_label",
        ],
        columns="question_key",
        values="Response",
        aggfunc="first"
    )
    .reset_index()
)

expected_cols = {
    "accurate_response",
    "confidence_response",
    "familiarity_response",
}

missing = expected_cols - set(tidy_df.columns)
if missing:
    raise ValueError(f"Missing expected response columns: {missing}")




# ---------------- Normalize accurate_response ----------------

valid_map = {
    "Yes": "SUPPORTS",
    "No": "REFUTES",
}

def normalize_accurate_response(value):
    if pd.isna(value):
        return pd.NA

    if value not in valid_map:
        raise ValueError(f"Invalid accurate_response value: {value}")

    return valid_map[value]

tidy_df["accurate_response"] = (
    tidy_df["accurate_response"]
    .apply(normalize_accurate_response)
)






display(tidy_df.head())





In [ ]:
# ---------------- Clean confidence & familiarity responses ----------------

def extract_int(value):
    if pd.isna(value):
        return pd.NA
    try:
        return int(str(value).split()[0])
    except ValueError:
        raise ValueError(f"Cannot convert value to int: {value}")

tidy_df["confidence_response"] = (
    tidy_df["confidence_response"]
    .apply(extract_int)
    .astype("Int64")
)

tidy_df["familiarity_response"] = (
    tidy_df["familiarity_response"]
    .apply(extract_int)
    .astype("Int64")
)


# ---------------- Filter out returned annotators ----------------

tidy_df = tidy_df[
    ~tidy_df["Annotator_ID"].isin(ANNOTATOR_ID_RETURNED)
].reset_index(drop=True)

print(
    f"Filtered annotators {ANNOTATOR_ID_RETURNED}. "
    f"Remaining rows: {len(tidy_df)}"
)

display(tidy_df.head())

tidy_df.to_csv(OUTPUT_FILEPATH, index=False)
